# Assignment: Generative Modeling with Gaussian Mixture Models (GMM)
## Using GMM to Learn and Generate Handwritten Digit Images

**Course:** Intro to Machine Learning  
**Topic:** Clustering Part 2 - GMM as a Generative Model  

---

### Objective

In this assignment, you will:
1. Load and explore the MNIST handwritten digit dataset
2. Apply PCA to reduce dimensionality while preserving variance
3. Fit a Gaussian Mixture Model (GMM) to clusters of digit images
4. Use the trained GMM to **generate brand new synthetic digit images**
5. Experiment with per-digit GMM models for improved generation quality

### Background

GMM learns the underlying probability distribution of data by fitting a mixture of Gaussian (normal) distributions. Once trained, we can **sample** from that learned distribution to create new, synthetic data that follows the same patterns as the original. This is the core idea behind **generative models** - the same family of concepts that powers things like GANs and diffusion models.

### Grading
- Part 1 - Data Loading & Exploration: **10 marks**
- Part 2 - PCA Dimensionality Reduction: **15 marks**
- Part 3 - Fit GMM to All Digits: **20 marks**
- Part 4 - Generate New Digit Images: **20 marks**
- Part 5 - Per-Digit GMM Generation: **25 marks**
- Part 6 - Reflection Questions: **10 marks**

**Total: 100 marks**

---
## Part 0 - Setup (Provided)

Run the cells below to import the required libraries and define helper functions. **Do not modify these cells.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import math
import matplotlib as mpl

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Helper function to display a single digit image
def showDigit(digit, label, size=28):
    some_digit_image = np.array(digit).reshape(size, size)
    plt.imshow(some_digit_image, cmap=mpl.cm.binary)
    plt.title(label)
    plt.axis("off")
    plt.show()

# Helper function to display multiple digit images in a grid
def showDigits(digits, labels, indexes, size=28, cols=6):
    pics = len(indexes)
    rows = math.ceil(pics / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 6))
    plt.axis("off")
    for i in range(0, pics):
        n = indexes[i]
        some_digit = digits[n:n+1]
        some_digit_image = np.array(some_digit).reshape(size, size)
        ax = axes[i // cols, i % cols] if rows > 1 else axes[i % cols]
        ax.axis("off")
        ax.imshow(some_digit_image, cmap=mpl.cm.binary)
        ax.set_title('Ind: {} - Lbl: {}'.format(indexes[i], labels[n]))
    plt.tight_layout()
    plt.axis("off")
    plt.show()

---
## Part 1 - Load and Explore the Data (10 marks)

We will use the **MNIST** dataset, which contains 28x28 pixel grayscale images of handwritten digits (0-9). Each image is stored as a flat array of 784 pixel values.

### Task 1.1 - Load the Data (5 marks)
Load the MNIST dataset using `fetch_openml`. Use only the **first 15,000 samples** for speed. Store the features in `X` and the labels in `y`. Print the shapes of both.

In [ ]:
# TODO: Load the MNIST dataset
# Hint: from sklearn.datasets import fetch_openml
#       mnist = fetch_openml('mnist_784', version=1)
#       Then slice the first 15000 rows for X and y

from sklearn.datasets import fetch_openml

# YOUR CODE HERE
X = ...
y = ...

print("X shape:", X.shape)
print("y shape:", y.shape)

### Task 1.2 - Visualize Some Digits (5 marks)
Use the provided `showDigits()` function to display at least **10 sample images** from the dataset. Choose a variety of indexes so you see different digits.

In [ ]:
# TODO: Display at least 10 sample digit images using showDigits()
# Hint: showDigits(X, y, [list_of_indexes])

# YOUR CODE HERE


---
## Part 2 - PCA Dimensionality Reduction (15 marks)

The raw MNIST data has 784 features (pixels). Fitting a GMM directly on 784 dimensions would be extremely slow and may not converge well. We use **PCA** to reduce the dimensionality while keeping most of the information.

### Task 2.1 - Apply PCA (10 marks)
Apply PCA to `X`, keeping **99% of the variance**. Store the transformed data in a variable called `data`. Print the new shape to see how many components PCA selected.

**Note:** The `whiten` parameter normalizes the variance of each component. You may set it to `False` for now.

In [ ]:
# TODO: Apply PCA with 0.99 variance retained
# Hint: pca = PCA(0.99, whiten=False)
#       data = pca.fit_transform(X)

# YOUR CODE HERE
pca = ...
data = ...

print("Original shape:", X.shape)
print("Reduced shape:", data.shape)
print("Number of components kept:", pca.n_components_)

### Task 2.2 - Interpret PCA (5 marks)
**Answer in a markdown cell below:**
- How many components did PCA keep?
- Why is PCA helpful before fitting a GMM? (Think about both speed and the curse of dimensionality.)

**YOUR ANSWER HERE:**

*(Double-click to edit this cell)*

---
## Part 3 - Fit a GMM to All Digits (20 marks)

Now we will fit a Gaussian Mixture Model to the PCA-reduced data. The GMM will learn the distribution of **all** digit images together.

### Task 3.1 - Create and Fit the GMM (10 marks)
Create a `GaussianMixture` model with:
- **300 components** (`n_components=300`)
- **`covariance_type='full'`** (allows each cluster to take any ellipsoidal shape)
- **`random_state=0`** (for reproducibility)

Fit it on the `data` (PCA-transformed). Print whether the model converged.

In [ ]:
# TODO: Create and fit a GaussianMixture model
# Hint: gmm = GaussianMixture(300, covariance_type='full', random_state=0)
#       gmm.fit(data)

# YOUR CODE HERE
gmm = ...


print("Model converged:", gmm.converged_)

### Task 3.2 - Why 300 Components? (10 marks)

**Answer in a markdown cell below:**
- Why might we use 300 components even though there are only 10 different digits (0-9)?
- What does each component represent in this context?
- How could you use AIC or BIC to choose the number of components? (You do not need to run AIC/BIC - just explain the concept.)

**YOUR ANSWER HERE:**

*(Double-click to edit this cell)*

---
## Part 4 - Generate New Digit Images (20 marks)

This is where the magic happens. The GMM has learned the distribution of digit images. We can now **sample** from it to create brand new, never-before-seen digit images.

### Task 4.1 - Sample from the GMM (10 marks)
Use the `.sample()` method to generate **10 new samples** from the fitted GMM. Then use `pca.inverse_transform()` to convert them back to 784-dimensional pixel space so they can be displayed as images.

In [ ]:
# TODO: Generate 10 new digit samples from the GMM
# Step 1: data_new = gmm.sample(10) - this returns a tuple, take the first element [0]
# Step 2: Use pca.inverse_transform() to get back to pixel space
# Step 3: Display using showDigits()

# YOUR CODE HERE
data_new = ...
data_new = data_new[0]  # sample() returns (samples, labels), we want samples

digits_new = ...  # inverse transform back to pixel space

# Display the generated digits
showDigits(digits=digits_new,
           labels=["","","","","","","","","",""],
           indexes=[1,2,3,4,5,6,7,8,9,0],
           size=28, cols=5)

### Task 4.2 - Analyze the Results (10 marks)
**Answer in a markdown cell below:**
- Do the generated images look like real handwritten digits?
- Are some clearer than others? Why might that be?
- Why might the images appear blurry or noisy? (Think about what the GMM is doing - it learned ALL digits at once.)

**YOUR ANSWER HERE:**

*(Double-click to edit this cell)*

---
## Part 5 - Per-Digit GMM Generation (25 marks)

The GMM above tried to model all 10 digits simultaneously. We can do better by training **separate GMMs for individual digits**. This way, each model only needs to learn the patterns of one specific digit.

### Task 5.1 - Train a GMM on a Single Digit (10 marks)
Pick a digit (0-9) and filter the PCA-transformed data to only include samples of that digit. Then fit a new GMM (300 components, full covariance) on just that subset. Generate and display 10 new samples.

**Hint:** You can filter with `data[y.astype(int) == YOUR_DIGIT]`

In [ ]:
# TODO: Choose a digit, filter data, fit GMM, generate and display new images

DIGIT_TO_GENERATE = ...  # Pick a digit 0-9

# Step 1: Filter the PCA data to only this digit
digit_data = ...

print(f"Number of samples for digit {DIGIT_TO_GENERATE}:", digit_data.shape[0])

# Step 2: Fit a new GMM on just this digit's data
gmm_single = ...


# Step 3: Generate 10 new samples
data_new_single = ...
data_new_single = data_new_single[0]

# Step 4: Inverse transform and display
new_digits_single = pca.inverse_transform(data_new_single)
showDigits(digits=new_digits_single,
           labels=["","","","","","","","","",""],
           indexes=[1,2,3,4,5,6,7,8,9,0],
           size=28, cols=5)

### Task 5.2 - Generate a Second Digit (5 marks)
Repeat the process above for a **different** digit. Display the generated samples.

In [ ]:
# TODO: Repeat for a different digit

DIGIT_TO_GENERATE_2 = ...  # Pick a DIFFERENT digit

# YOUR CODE HERE



### Task 5.3 - Compare Quality (10 marks)
**Answer in a markdown cell below:**
- Compare the images generated from the per-digit GMM (Part 5) to those from the all-digit GMM (Part 4). Which are clearer? Why?
- What is the trade-off of training 10 separate models vs. one combined model?
- In real-world applications, how is this idea of learning data distributions and generating new samples used? Give at least one example beyond handwritten digits.

**YOUR ANSWER HERE:**

*(Double-click to edit this cell)*

---
## Part 6 - Reflection Questions (10 marks)

Answer the following questions in the markdown cell below. Each is worth 5 marks.

1. **Generative vs. Discriminative:** In your own words, explain the difference between a *generative* model (like GMM) and a *discriminative* model (like logistic regression). What does each type of model learn?

2. **Role of PCA:** What would happen if we skipped the PCA step and fit the GMM directly on the 784-dimensional pixel data? Consider both practical issues (speed, convergence) and conceptual issues (noise, meaningful features).

**YOUR ANSWER HERE:**

*(Double-click to edit this cell)*

---
### Submission

- Save this notebook with all cells executed (Kernel -> Restart & Run All)
- Ensure all outputs (plots, printed shapes) are visible
- Submit the `.ipynb` file

**Good luck!**